# Reliability: Retries & Fallback

Models and tools occasionally fail (rate limits, timeouts, flaky APIs). These middleware make an agent **resilient** without you writing try/except everywhere:

| Middleware | What it does |
| --- | --- |
| `ToolRetryMiddleware` | Re-runs a failing **tool** with backoff |
| `ModelRetryMiddleware` | Re-runs a failing **model** call with backoff |
| `ModelFallbackMiddleware` | Switches to a **backup model** when the primary fails |

In [1]:
%run ../../langchain_common.py

C:\Users\akhawaja\git\cs4603\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


## 1. Retrying a flaky tool

`check_inventory` below fails the first two times (simulating a flaky API). `ToolRetryMiddleware` retries it automatically until it succeeds.

In [2]:
from langchain.agents.middleware import ToolRetryMiddleware

inventory_attempts = {"count": 0}


@tool
def check_inventory(product: str) -> str:
    """Check stock for a product (simulated flaky external API)."""
    inventory_attempts["count"] += 1
    if inventory_attempts["count"] < 3:
        raise RuntimeError(f"Temporary API error (attempt {inventory_attempts['count']})")
    return f"{product}: 42 units in stock."


retry_agent = create_agent(
    model=llm_noreason,
    tools=[check_inventory],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            initial_delay=0.0,   # no waiting, keeps the demo fast
            backoff_factor=1.0,
            jitter=False,
        )
    ],
)

In [3]:
inventory_attempts["count"] = 0

resp = retry_agent.invoke(
    {"messages": [HumanMessage(content="How many units of widget-X are in stock?")]}
)
for m in resp["messages"]:
    m.pretty_print()

print("\nTotal tool attempts (2 failures + 1 success):", inventory_attempts["count"])

================================ Human Message =================================

How many units of widget-X are in stock?
================================== Ai Message ==================================
Tool Calls:
  check_inventory (call_8d5e0afd-ee09-4527-b0c0-8ab5059ff3af)
 Call ID: call_8d5e0afd-ee09-4527-b0c0-8ab5059ff3af
  Args:
    product: widget-X
================================= Tool Message =================================
Name: check_inventory

widget-X: 42 units in stock.
================================== Ai Message ==================================

There are 42 units of widget-X in stock.

Total tool attempts (2 failures + 1 success): 3


## 2. Retrying the model

`ModelRetryMiddleware` works the same way for model calls. Here it is configured on a normal agent; if a model call raised a transient error it would be retried up to `max_retries` times before giving up.

In [4]:
from langchain.agents.middleware import ModelRetryMiddleware

resilient_agent = create_agent(
    model=llm_noreason,
    tools=[],
    middleware=[
        ModelRetryMiddleware(
            max_retries=2,
            retry_on=(Exception,),
            initial_delay=0.0,
            backoff_factor=1.0,
            jitter=False,
        )
    ],
)

resilient_agent.invoke(
    {"messages": [HumanMessage(content="Say hello in one short sentence.")]}
)["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hello!


## 3. Falling back to a backup model

`ModelFallbackMiddleware` lists one or more **backup models**. If the agent's primary model fails, the request is retried against each fallback in order.

Below the primary model name is intentionally invalid, so every primary call fails and the agent automatically falls back to a working model.

In [5]:
from langchain.agents.middleware import ModelFallbackMiddleware

broken_primary = create_noreason_llm(model="this-model-does-not-exist")

fallback_agent = create_agent(
    model=broken_primary,
    tools=[],
    middleware=[
        ModelFallbackMiddleware(llm_noreason),  # known-good backup
    ],
)

fallback_agent.invoke(
    {"messages": [HumanMessage(content="What is 2 + 2? Answer briefly.")]}
)["messages"][-1].pretty_print()

================================== Ai Message ==================================

2 + 2 equals 4.


## Summary

- `ToolRetryMiddleware` / `ModelRetryMiddleware` retry with exponential backoff (`initial_delay`, `backoff_factor`, `jitter`).
- `retry_on` limits retries to specific exception types; `on_failure` controls what happens after the last attempt.
- `ModelFallbackMiddleware` swaps in backup models when the primary keeps failing.
- Combine these with the usage-limit guardrails from the guardrails notebook for production-grade reliability.